In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

### Module 4.1 (Moment 1): The Theory of Analytical Graphics

#### The Core Problem: Control vs. Simplicity

The entire Python visualization world is built on a fundamental tension:

1.  **Matplotlib:** A low-level, powerful, and *verbose* library. It gives you 100% control over every "artist" (every line, point, and piece of text). This is the **Pythonista's** tool for control.
2.  **Seaborn:** A high-level, statistically-aware library. It gives you 100% of the *statistical logic* (automatically calculating regressions, CIs, histograms, groupings). This is the **Statistician's** tool for insight.

An amateur uses one or the other. A professional uses *both at the same time*. Our goal in this module is to learn how to **force Matplotlib's control onto Seaborn's statistical power.**

#### Part 1: The Matplotlib Foundation (The Two APIs)

To control Seaborn, you must first understand the architecture of Matplotlib. Matplotlib has two "APIs" or ways of being used. This is the single most important concept.

**1. The `pyplot` (State-Machine) API**
This is what 90% of beginners learn. It works like a simple notebook. Matplotlib keeps track of *one* "active" plot, and all commands (`plt.title`, `plt.plot`) are applied to it.

  * **Analogy:** You have *one* piece of paper on your desk. `plt.plot()` draws on it. `plt.title()` writes on it.
  * **Code Example:**
    ```python
    plt.plot([1, 2, 3], [10, 20, 15]) # Draws on the "active" plot
    plt.title("My Plot")              # Writes on the "active" plot
    plt.show()
    ```
  * **Limitation:** This is fine for one plot, but it becomes a confusing nightmare for building a dashboard. How do you tell it which of your four subplots is "active"? It becomes unwieldy.

**2. The Object-Oriented (OO) API (The Professional's Way)**
This is our focus. You *never* let Matplotlib guess what plot is active. You *explicitly* create your canvas (the `Figure`) and your subplots (the `Axes`) as objects. You then call methods *directly* on those objects.

  * **Analogy:** You are a manager. You create a `Figure` (the dashboard canvas) and then you create `Axes` objects (e.g., `ax1`, `ax2`) for each subplot. You give explicit commands: "ax1, set your title to 'Age Distribution'." "ax2, draw this scatter plot."

  * **The Golden Code (`plt.subplots`)**: This is the command that creates both the canvas and the subplots at the same time.

    ```python
    # Create a figure (fig) and ONE subplot (ax)
    fig, ax = plt.subplots(figsize=(8, 6))

    # Now, call methods on the 'ax' object
    ax.plot([1, 2, 3], [10, 20, 15])
    ax.set_title("My Plot (OO API)")
    ax.set_xlabel("Time")
    plt.show()
    ```

  * **For Dashboards (Our Goal):**

    ```python
    # Create a 2x2 dashboard
    # 'fig' is the whole canvas
    # 'axes' is a 2D NumPy array of the four subplots
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 10))

    # Access the subplots by their index
    axes[0, 0].set_title("Top-Left Plot")
    axes[0, 1].set_title("Top-Right Plot")
    axes[1, 0].set_title("Bottom-Left Plot")
    axes[1, 1].set_title("Bottom-Right Plot")

    plt.show()
    ```

**Why this is advanced:** This OO approach is the *only* way to build complex, multi-plot figures. It is the key to unlocking dashboard-style analysis.

#### Part 2: The Seaborn Philosophy (Two Kinds of Functions)

Now that we have our `ax` objects, we can see how Seaborn interacts with them. Seaborn functions are *not* all the same. They fall into two distinct families.

**1. Axes-Level Functions**

  * **What they are:** `scatterplot()`, `boxplot()`, `histplot()`, `regplot()`, `pointplot()`, `kdeplot()`, `violinplot()`, etc.

  * **Key Property:** They are designed to draw *onto a specific Matplotlib `Axes`*.

  * **How you use them:** You pass your `ax` object to them using the **`ax=` parameter**.

  * **This is the key:** This is the "golden bridge" that connects Seaborn's statistical power to Matplotlib's control.

    ```python
    fig, ax = plt.subplots()
    sns.histplot(data=df, x='age', kde=True, ax=ax) # Tell Seaborn to draw on 'ax'
    ax.set_title("My Custom Title")
    ```

**2. Figure-Level Functions**

  * **What they are:** `jointplot()`, `pairplot()`, `FacetGrid()`, `clustermap()`.
  * **Key Property:** They are *not* designed to draw on your existing `ax`. They *create and manage their own* `Figure` and `Axes` grid.
  * **Why:** A `jointplot` is not one plot; it is *three* (the main scatter, and two marginal histograms). A `pairplot` is a complex grid. These functions *are* the dashboard.
  * **Limitation:** Because they "own" the figure, they are harder to customize. You cannot easily put a `jointplot` into the top-left corner of your `plt.subplots(2, 2)` grid.

**The Statistician-Pythonista's Rule:**

> You will primarily use **Axes-level functions** (`scatterplot`, `histplot`, etc.) and pass them to the **Matplotlib OO API (`ax=`)**. This gives you maximum statistical power *and* maximum layout control. You use Figure-level functions (`pairplot`) for quick, exploratory "first looks".

#### Part 3: The Statistician's Toolbox (Beyond Basic Plots)

Your `plot_seaborn` function is great, but it treats all plots as equal. A true statistician knows *why* to pick one plot over another for its inferential power.

**1. Visualizing Inference (Not Just Data)**
You should not just plot points. You should plot *what the points imply*.

  * **`sns.regplot()` (Regression Plot):**

      * **What it is:** This is a `scatterplot` *plus* a linear regression.
      * **What it does:** It automatically:
        1.  Runs a fast linear regression ($y \sim x$).
        2.  Plots the resulting regression line.
        3.  Calculates and plots the 95% confidence interval for that regression line (the light-colored "cone" around the line).
      * **Statistical Read:** If the confidence interval "cone" is very wide, it means the relationship is weak and you have low confidence in the slope. If it is tight, the relationship is strong.

  * **`sns.pointplot()` (Point Plot):**

      * **What it is:** This chart looks like a simple line chart, but it is for *categorical* data.
      * **What it does:** The dot is the `mean` (or other estimator) for that category. The vertical line is the **95% confidence interval for the mean**.
      * **Statistical Read:** This is far superior to a `barplot`. A `barplot` only shows the mean. A `pointplot` shows the mean *and* its uncertainty. If the confidence intervals for two categories do not overlap, you have strong evidence that their means are *statistically significantly different*.

**2. Visualizing Distributions (Rigorously)**
You already know `histplot` and `boxplot`.

  * **`sns.violinplot()`:**

      * **What it is:** The perfect marriage of a `boxplot` and a `kdeplot`.
      * **What it does:** It shows a `boxplot` in the center, and on both sides, it shows the full density (the "shape") of the data.
      * **Statistical Read:** It answers questions a `boxplot` cannot. For example, a `boxplot` might look symmetric, but the `violinplot` could reveal that the data is **bimodal** (has two peaks).

  * **`sns.ecdfplot()` (Empirical Cumulative Distribution Function):**

      * **What it is:** A very powerful and often superior alternative to a histogram.
      * **What it does:** Instead of bins, it plots a step function. The Y-axis is "what percentage of my data is less than or equal to this X-value".
      * **Statistical Read:** It is fantastic for reading percentiles. To find the median (50th percentile), you just find Y=0.50 and look at the corresponding X. It also shows skew and modality clearly, without the "binning bias" of a histogram.

**3. Visualizing Multi-Variable Relationships (Grids)**
These are the Figure-level functions we discussed.

  * **`sns.pairplot(df, hue='category')`:**
      * **What it is:** The single most powerful "first look" tool in all of data science. It creates a matrix of scatterplots for every numeric variable against every other, and histograms/KDEs on the diagonal.
      * **Statistical Read:** You can diagnose *all* pairwise relationships (linear, non-linear, correlated, uncorrelated) and *all* distributions in one command.
  * **`sns.jointplot(x=..., y=..., kind='kde'/'hex')`:**
      * **What it is:** A deep dive into *two* variables. It shows the main `scatterplot` (or a `kde` or `hexbin` plot for overplotting) and the individual histograms/KDEs for both X and Y in the margins.
  * **`sns.clustermap(df.corr())`:**
      * **What it is:** A `heatmap` *plus* hierarchical clustering.
      * **Statistical Read:** It automatically re-orders your correlation matrix so that highly-correlated features are grouped together, revealing the hidden "blocks" of co-related variables in your dataset.

#### Part 4: Professional Polish (Color, Context, and Annotation)

1.  **Color as Information (Palettes):** Color is not decoration; it is an aesthetic, just like `x` and `y`.

      * **Categorical (`palette='Set1'` or `'tab10'`):** For distinct, non-ordered groups (e.g., `Country`: USA, Brazil, UK). Use high-contrast, different colors.
      * **Sequential (`palette='Blues'` or `'viridis'`):** For numeric data that goes from low-to-high (e.g., `Price`: 0 to 100). Use a gradient of a single color.
      * **Diverging (`palette='coolwarm'` or `'vlag'`):** For numeric data with a *meaningful center* (like 0). (e.g., `Correlation`: -1 to +1, or `Profit`: -50 to +50). Use two different colors that diverge from a neutral center.

2.  **Context (`sns.set_context()`):**

      * This scales all plot elements (text, lines) for its final destination.
      * `sns.set_context('notebook')`: Default.
      * `sns.set_context('paper')`: Smaller, for academic papers.
      * `sns.set_context('talk')`: Bolder, for presentations.
      * `sns.set_context('poster')`: Huge, for posters.

3.  **Annotation (`ax.annotate()`):**

      * This is how you *tell the story*. A plot shows *what*, an annotation tells the *so what*.
      * You use it to add your statistical findings (like the $r^2$ value from a regression) or to point an arrow at a specific, interesting outlier.

